# Inspect the adapters
Teaching notebook; run with the installed project environment. Outputs are intentionally unexecuted, not fabricated. Read learning/COMPLETE_LEARNING_GUIDE.md first.

In [ ]:
import copy
import torch
from torch import nn
from lowrank_lab.adapters import LoRALinear, DoRALinear, PiSSALinear
base = nn.Linear(3, 2, bias=False)
base.weight.data.copy_(torch.tensor([[3.,4.,0.],[0.,0.,2.]]))
x = torch.tensor([[1.,0.,0.]])
print(base(x))

In [ ]:
for cls in [LoRALinear, DoRALinear, PiSSALinear]:
    layer = cls(copy.deepcopy(base), rank=1, alpha=2)
    print(cls.__name__, layer(x), layer.A.shape, layer.B.shape)
    torch.testing.assert_close(layer(x), base(x))

In [ ]:
layer = LoRALinear(copy.deepcopy(base), rank=1, alpha=2)
optimizer = torch.optim.SGD([p for p in layer.parameters() if p.requires_grad], lr=0.01)
for step in range(2):
    optimizer.zero_grad()
    layer(x).square().sum().backward()
    print(step, layer.A.grad, layer.B.grad)
    optimizer.step()